In [3]:
import pandas as pd
import os
from dotenv import load_dotenv
from datasets import Dataset
from pandas import DataFrame

# Updated Ragas metric imports
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas import evaluate

# LangChain OpenAI integrations
from ragas.llms import LangchainLLMWrapper
# Use Groq for the Judge and HuggingFace for fast, local embeddings
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
df = pd.read_csv("sample_dataset.csv")
display(df)

,question,context,answer,reference,scenario
0,What is Kubernetes?,Kubernetes is an open-source container orchest...,Kubernetes is an open-source container orchest...,Kubernetes is an open-source container orchest...,A
1,Who created Docker?,Docker was created by Solomon Hykes.,Docker was created by Solomon Hykes.,Docker was created by Solomon Hykes.,A
2,What is LiteLLM?,LiteLLM provides a unified interface to multip...,LiteLLM provides a unified interface to multip...,LiteLLM provides a unified interface to multip...,A
3,What is Qdrant?,Qdrant is an open-source vector database.,Qdrant is an open-source vector database.,Qdrant is an open-source vector database.,A
4,What is Kubernetes?,Tomatoes grow well in summer. Redis is an in-m...,Kubernetes is an open-source container orchest...,Kubernetes is an open-source container orchest...,B
5,Who created Docker?,LangFuse provides observability. PostgreSQL is...,Docker was created by Solomon Hykes.,Docker was created by Solomon Hykes.,B
6,What is RAG?,Anthropic develops Claude models. Redis is use...,RAG combines retrieval and generation.,Retrieval Augmented Generation combines retrie...,B
7,What is Ollama?,Qdrant is a vector database. PostgreSQL is rel...,Ollama runs local language models.,Ollama enables running large language models l...,B
8,What is Kubernetes?,Kubernetes is open-source.,Kubernetes is an open-source container orchest...,Kubernetes is an open-source container orchest...,C
9,Who created Docker?,Docker was created by Solomon Hykes.,Docker was created by Solomon Hykes and starte...,Docker was created by Solomon Hykes and led to...,C


In [5]:
dataset = Dataset.from_dict({
    "question": df["question"].tolist(),

    "answer": df["answer"].tolist(),

    "contexts": [
        [c]
        for c in df["context"].tolist()
    ],

    "reference": df["reference"].tolist()

})

display(DataFrame(dataset))

,question,answer,contexts,reference
0,What is Kubernetes?,Kubernetes is an open-source container orchest...,[Kubernetes is an open-source container orches...,Kubernetes is an open-source container orchest...
1,Who created Docker?,Docker was created by Solomon Hykes.,[Docker was created by Solomon Hykes.],Docker was created by Solomon Hykes.
2,What is LiteLLM?,LiteLLM provides a unified interface to multip...,[LiteLLM provides a unified interface to multi...,LiteLLM provides a unified interface to multip...
3,What is Qdrant?,Qdrant is an open-source vector database.,[Qdrant is an open-source vector database.],Qdrant is an open-source vector database.
4,What is Kubernetes?,Kubernetes is an open-source container orchest...,[Tomatoes grow well in summer. Redis is an in-...,Kubernetes is an open-source container orchest...
5,Who created Docker?,Docker was created by Solomon Hykes.,[LangFuse provides observability. PostgreSQL i...,Docker was created by Solomon Hykes.
6,What is RAG?,RAG combines retrieval and generation.,[Anthropic develops Claude models. Redis is us...,Retrieval Augmented Generation combines retrie...
7,What is Ollama?,Ollama runs local language models.,[Qdrant is a vector database. PostgreSQL is re...,Ollama enables running large language models l...
8,What is Kubernetes?,Kubernetes is an open-source container orchest...,[Kubernetes is open-source.],Kubernetes is an open-source container orchest...
9,Who created Docker?,Docker was created by Solomon Hykes and starte...,[Docker was created by Solomon Hykes.],Docker was created by Solomon Hykes and led to...


In [6]:
# 1. Load the environment variables from your .env file
load_dotenv()

# Verify the key is loaded (don't print the actual key in output!)
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found. Please check your .env file.")

# 2. Initialize Models
# Requires GROQ_API_KEY in your .env file
judge_model = ChatGroq(
    # CHECK: https://console.groq.com/docs/deprecations
    model="llama-3.1-8b-instant",
    #model="llama-3.1-70b-versatile",
    temperature=0,
    max_tokens=4096
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(f"✅ Judge LLM configured as: {judge_model.model_name}")
print(f"✅ Embeddings configured as: {embeddings.model_name}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Judge LLM configured as: llama-3.1-8b-instant
✅ Embeddings configured as: sentence-transformers/all-MiniLM-L6-v2


In [7]:
# Execute Ragas using the cloud models
score = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=judge_model,
    embeddings=embeddings,
    # Prevents the entire run from failing if one evaluation errors out
    raise_exceptions=False,
    # Prevents overload
    batch_size=4
)

# Output as a clean Pandas DataFrame for analysis
results_df = score.to_pandas()
display(DataFrame(results_df))
print()

Evaluating:   0%|          | 0/64 [00:00<?, ?it/s]

Batch 1/16:   0%|          | 0/4 [00:00<?, ?it/s]

Exception raised in Job[8]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Exception raised in Job[56]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Kubernetes?,[Kubernetes is an open-source container orches...,Kubernetes is an open-source container orchest...,Kubernetes is an open-source container orchest...,1.0,1.000000,1.0,1.000000
1,Who created Docker?,[Docker was created by Solomon Hykes.],Docker was created by Solomon Hykes.,Docker was created by Solomon Hykes.,1.0,1.000000,1.0,1.000000
2,What is LiteLLM?,[LiteLLM provides a unified interface to multi...,LiteLLM provides a unified interface to multip...,LiteLLM provides a unified interface to multip...,NaN,0.876897,1.0,1.000000
3,What is Qdrant?,[Qdrant is an open-source vector database.],Qdrant is an open-source vector database.,Qdrant is an open-source vector database.,1.0,1.000000,1.0,1.000000
4,What is Kubernetes?,[Tomatoes grow well in summer. Redis is an in-...,Kubernetes is an open-source container orchest...,Kubernetes is an open-source container orchest...,0.0,1.000000,0.0,0.000000
5,Who created Docker?,[LangFuse provides observability. PostgreSQL i...,Docker was created by Solomon Hykes.,Docker was created by Solomon Hykes.,0.0,1.000000,0.0,0.000000
6,What is RAG?,[Anthropic develops Claude models. Redis is us...,RAG combines retrieval and generation.,Retrieval Augmented Generation combines retrie...,0.0,0.832555,0.0,0.000000
7,What is Ollama?,[Qdrant is a vector database. PostgreSQL is re...,Ollama runs local language models.,Ollama enables running large language models l...,0.0,0.798548,0.0,0.000000
8,What is Kubernetes?,[Kubernetes is open-source.],Kubernetes is an open-source container orchest...,Kubernetes is an open-source container orchest...,0.0,1.000000,0.0,0.500000
9,Who created Docker?,[Docker was created by Solomon Hykes.],Docker was created by Solomon Hykes and starte...,Docker was created by Solomon Hykes and led to...,0.5,1.000000,0.0,0.500000


In [35]:
avg_faithfulness = \
    results_df["faithfulness"].mean()

avg_relevance = \
    results_df["answer_relevancy"].mean()

avg_ctx_precision = \
    results_df["context_precision"].mean()

avg_ctx_recall = \
    results_df["context_recall"].mean()

print("Faithfulness :", round(avg_faithfulness,2))
print("Relevance    :", round(avg_relevance,2))
print("Context Precision :", round(avg_ctx_precision,2))
print("Context Recall    :", round(avg_ctx_recall,2))

Faithfulness : 0.5
Relevance    : 0.9
Context Precision : 0.56
Context Recall    : 0.56


In [36]:
results_df.sort_values(
    by="faithfulness"
).head(5)

results_df.to_csv(
    "evaluation_results.csv",
    index=False
)